# PPDONet-Time-Stan: Velocidad Azimutal

Version espacio-temporal del modelo para velocidad azimutal `v_theta`.

En esta variante, los parametros fisicos (`ALPHA`, `PLANETMASS`, `ASPECTRATIO`) siguen entrando por la red branch. La red trunk recibe ahora la coordenada espacio-temporal transformada `(r_scaled, sin(theta), cos(theta), time_scaled)`.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

models_time_dir = Path.cwd() / "Models_time"
if (models_time_dir / "ppdonet_common.py").exists():
    sys.path.insert(0, str(models_time_dir))
elif (Path.cwd() / "ppdonet_common.py").exists():
    sys.path.insert(0, str(Path.cwd()))

sys.modules.pop("ppdonet_common", None)
from ppdonet_common import *

device = get_device()
print_device_info(device)


## Datos

In [ ]:
train_dir, test_dir = download_planet2disk_data()
print("Train dir:", train_dir)
print("Test dir :", test_dir)

ds_sigma_tr, ds_vr_tr, ds_vtheta_tr = open_outputs(train_dir)
ds_sigma_te, ds_vr_te, ds_vtheta_te = open_outputs(test_dir)

params_tr, where_tr = find_and_load_params(train_dir, device)
params_te, where_te = find_and_load_params(test_dir, device)
print(where_tr)
print("params_tr:", params_tr.shape, params_tr.device)


## Coordenadas espacio-temporales

In [ ]:
coords_tf, r_vals, th_vals, time_vals = build_coords(ds_sigma_tr, device)
print("coords_tf:", coords_tf.shape, coords_tf.device)
print("time_vals:", time_vals.shape, time_vals[: min(5, len(time_vals))])


## Dataset y loaders

In [ ]:
vtheta_var = list(ds_vtheta_tr.data_vars)[0]

# None usa todos los puntos (time, r, theta). Para entrenamientos rapidos, usar un entero menor.
NPOINTS_SAMPLE = None
BATCH_SIZE = 1

train_vt_loader, test_vt_loader = make_field_loaders(
    ds_vtheta_tr,
    ds_vtheta_te,
    params_tr,
    params_te,
    coords_tf,
    vtheta_var,
    time_vals,
    device,
    log_target=False,
    subtract_background=False,
    n_points_sample=NPOINTS_SAMPLE,
    batch_size=BATCH_SIZE,
)

u0, y0, t0 = next(iter(train_vt_loader))
print("sample shapes (u, y, target):", u0.shape, y0.shape, t0.shape)
print("devices:", u0.device, y0.device, t0.device)


## Entrenamiento

In [ ]:
EPOCHS = 1500
LR = 1e-4
CHECKPOINT = "modelo_vazimuth_time_Stan.pt"

model_vt = build_ppdonet(act="stan", stan_beta=0.1, stan_positive_beta=True)
hist_vt, model_vt = train_model(
    model_vt,
    train_vt_loader,
    test_vt_loader,
    device,
    epochs=EPOCHS,
    lr=LR,
)
save_model(model_vt, CHECKPOINT)


## Evaluacion

In [ ]:
CHECKPOINT = "modelo_vazimuth_time_Stan.pt"
model_vt = load_model(CHECKPOINT, device)

mse_vt, r2_vt = evaluate_model_vtheta_dvx(model_vt, test_vt_loader, r_vals, device)
print(f"v_theta dvx: MSE={mse_vt:.4e}, R2={r2_vt:.4f}")


## Ejemplo de test

In [ ]:
idx_example = 0
time_index = 0

pred_vt_map, true_vt_map, mse_vt_ex, r2_vt_ex = predict_single_field(
    model_vt,
    params_te,
    coords_tf,
    ds_vtheta_te,
    vtheta_var,
    r_vals,
    th_vals,
    time_vals,
    device,
    idx_example=idx_example,
    time_index=time_index,
    log_target=False,
    subtract_background=False,
    fargo_kepler_delta=True,
)

print(f"v_theta ejemplo {idx_example}, time_index={time_index}: MSE={mse_vt_ex:.4e}, R2={r2_vt_ex:.4f}")


In [ ]:
plot_field_maps(
    pred_vt_map,
    true_vt_map,
    r_vals,
    th_vals,
    title=f"v_theta (delta kepleriano) - Test idx={idx_example} - time={time_vals[time_index]}",
    cmap="RdBu_r",
    robust=True,
)
